In [ ]:
"""
电除尘器系统辨识与前馈+反馈控制仿真模型（平滑优化版）
============================================================
在原有前馈控制的基础上，引入简单的 PI 反馈机制调整二次电压，
使仿真出口浓度更合理、更接近实际闭环效果。
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings("ignore")

plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

# ===========================================================================
# 1. 数据加载与预处理
# ===========================================================================
print("=" * 60)
print("加载数据...")

# 使用正确的文件路径
file_path = (
    r"C:\Users\Administrator\Desktop\数模校赛\题目发布\赛题\2026_A题\27FD7100.xlsx"
)

xl = pd.ExcelFile(file_path)
if "Cement_ESP_Data" in xl.sheet_names:
    df = pd.read_excel(file_path, sheet_name="Cement_ESP_Data")
else:
    df = pd.read_excel(file_path)

df = df.sort_values("timestamp").reset_index(drop=True)
df = df[(df["C_in_gNm3"] > 0) & (df["Q_Nm3h"] > 0)]

df["C_in_mg"] = df["C_in_gNm3"] * 1000.0
df["T_K"] = df["Temp_C"] + 273.15
print(f"有效数据量: {len(df)}")

# ===========================================================================
# 2. 动态积灰状态 S_i (基于原始振打周期, 仅分析用)
# ===========================================================================
print("\n构造动态积灰状态 S_i ...")
alpha_soot = 0.3
for i in range(1, 5):
    col = f"T{i}_s"
    S = np.zeros(len(df))
    S[0] = df[col].iloc[0]
    for t in range(1, len(df)):
        S[t] = alpha_soot * df[col].iloc[t] + (1 - alpha_soot) * S[t - 1]
    df[f"S{i}"] = S

# ===========================================================================
# 3. 前馈控制器训练 (入口条件 -> 操作参数)
# ===========================================================================
print("\n训练前馈控制器 ...")
ff_features = ["C_in_gNm3", "Q_Nm3h", "Temp_C"]
ff_targets = ["U1_kV", "U2_kV", "U3_kV", "U4_kV", "T1_s", "T2_s", "T3_s", "T4_s"]

ff_model = MultiOutputRegressor(
    GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42
    )
)
ff_model.fit(df[ff_features], df[ff_targets])

ff_pred_all = ff_model.predict(df[ff_features])
print("前馈控制器拟合效果 (R²):")
for i, col in enumerate(ff_targets):
    r2 = r2_score(df[col], ff_pred_all[:, i])
    print(f"  {col}: {r2:.3f}")

# ===========================================================================
# 4. 前馈输出平滑处理 (移动平均, 窗口=7)
# ===========================================================================
print("\n对前馈输出进行移动平均平滑 (窗口=7)...")
win = 7

U1_ff, U2_ff, U3_ff, U4_ff = (
    ff_pred_all[:, 0],
    ff_pred_all[:, 1],
    ff_pred_all[:, 2],
    ff_pred_all[:, 3],
)
T1_ff, T2_ff, T3_ff, T4_ff = (
    ff_pred_all[:, 4],
    ff_pred_all[:, 5],
    ff_pred_all[:, 6],
    ff_pred_all[:, 7],
)


def smooth(x, w):
    return pd.Series(x).rolling(w, center=True, min_periods=1).mean().values


U1_ff = smooth(U1_ff, win)
U2_ff = smooth(U2_ff, win)
U3_ff = smooth(U3_ff, win)
U4_ff = smooth(U4_ff, win)

T1_ff = smooth(T1_ff, win)
T2_ff = smooth(T2_ff, win)
T3_ff = smooth(T3_ff, win)
T4_ff = smooth(T4_ff, win)

# 电压物理截断
U_min, U_max = 30, 80
U1_ff = np.clip(U1_ff, U_min, U_max)
U2_ff = np.clip(U2_ff, U_min, U_max)
U3_ff = np.clip(U3_ff, U_min, U_max)
U4_ff = np.clip(U4_ff, U_min, U_max)

# 由平滑后的振打周期重新计算积灰状态 S_ff
S1_ff = np.zeros(len(df))
S2_ff = np.zeros(len(df))
S3_ff = np.zeros(len(df))
S4_ff = np.zeros(len(df))
S1_ff[0] = T1_ff[0]
S2_ff[0] = T2_ff[0]
S3_ff[0] = T3_ff[0]
S4_ff[0] = T4_ff[0]
for t in range(1, len(df)):
    S1_ff[t] = alpha_soot * T1_ff[t] + (1 - alpha_soot) * S1_ff[t - 1]
    S2_ff[t] = alpha_soot * T2_ff[t] + (1 - alpha_soot) * S2_ff[t - 1]
    S3_ff[t] = alpha_soot * T3_ff[t] + (1 - alpha_soot) * S3_ff[t - 1]
    S4_ff[t] = alpha_soot * T4_ff[t] + (1 - alpha_soot) * S4_ff[t - 1]

U_mat_ff = np.column_stack([U1_ff, U2_ff, U3_ff, U4_ff])
S_mat_ff = np.column_stack([S1_ff, S2_ff, S3_ff, S4_ff])
T_actual = df["Temp_C"].values
C_in_mg = df["C_in_mg"].values
Q_actual = df["Q_Nm3h"].values

# ===========================================================================
# 5. 物理模型定义与辨识 (MSE 损失)
# ===========================================================================
print("\n开始物理模型辨识（仿真出口浓度误差最小化，MSE损失）...")


def physical_omega(U_mat, S_mat, T_v, params):
    K, alpha, beta, k1, k2, k3, k4 = params
    T_K = T_v + 273.15
    k_arr = np.array([k1, k2, k3, k4])
    U_eff = U_mat - k_arr * S_mat
    U_eff = np.clip(U_eff, 1.0, None)
    sum_U = np.sum(U_eff, axis=1)
    Omega = K * (T_K ** (-beta)) * (sum_U**alpha)
    return Omega


def loss_identification(params):
    Omega_sim = physical_omega(U_mat_ff, S_mat_ff, T_actual, params)
    C_sim = C_in_mg * np.exp(-Omega_sim / Q_actual)
    return np.mean((C_sim - 50.0) ** 2)


bounds = [
    (1e3, 2e6),  # K
    (1.0, 2.5),  # alpha
    (0.0, 2.0),  # beta
    (0.0, 0.02),  # k1
    (0.0, 0.02),  # k2
    (0.0, 0.02),  # k3
    (0.0, 0.02),  # k4
]

result = differential_evolution(
    loss_identification, bounds, maxiter=200, popsize=25, seed=42, polish=True
)
opt_params = result.x

print(f"辨识完成，最终损失（MSE）= {result.fun:.4f}")
print(f"参数: K={opt_params[0]:.2f}, α={opt_params[1]:.3f}, β={opt_params[2]:.3f}")
for i in range(4):
    print(f"  k{i+1} = {opt_params[3+i]:.6f}")

# ===========================================================================
# 6. 原始开环前馈仿真结果（无反馈）
# ===========================================================================
Omega_ff = physical_omega(U_mat_ff, S_mat_ff, T_actual, opt_params)
C_sim_ff = C_in_mg * np.exp(-Omega_ff / Q_actual)

print("\n" + "=" * 50)
print("【无反馈】仅前馈控制仿真结果")
print(f"  仿真出口浓度均值: {np.mean(C_sim_ff):.2f} mg/Nm³")
print(f"  仿真出口浓度标准差: {np.std(C_sim_ff):.4f}")
print(f"  中位数: {np.median(C_sim_ff):.2f}")

# ===========================================================================
# 7. 引入简单 PI 反馈机制（前馈 + 反馈控制）
# ===========================================================================
print("\n" + "=" * 50)
print("引入 PI 反馈控制（前馈 + 出口浓度偏差反馈调节电压）...")

# PI 控制器参数（可调）
Kp = 0.1  # 比例增益
Ki = 0.005  # 积分增益
delta_U_max = 10.0  # 单次电压调整上限 (kV)

# 初始化
C_sim_fb = np.zeros(len(df))
U_mat_fb = np.zeros_like(U_mat_ff)
S_fb = np.zeros_like(S_mat_ff)  # 含反馈电压后的积灰状态（仍由振打周期递推）

# 反馈误差变量
e_prev = 0.0
I_err = 0.0

for t in range(len(df)):
    # 1) 由前馈振打周期计算当前积灰状态（与电压无关）
    if t == 0:
        S_fb[t] = [T1_ff[0], T2_ff[0], T3_ff[0], T4_ff[0]]
    else:
        for i in range(4):
            T_col = [T1_ff, T2_ff, T3_ff, T4_ff][i]
            S_fb[t, i] = alpha_soot * T_col[t] + (1 - alpha_soot) * S_fb[t - 1, i]

    # 2) 前馈电压 + PI 反馈修正量
    delta_U = Kp * e_prev + Ki * I_err
    delta_U = np.clip(delta_U, -delta_U_max, delta_U_max)  # 限制单次调整幅度
    U_fb = U_mat_ff[t] + delta_U  # 四个电场加上相同的电压调整量
    U_fb = np.clip(U_fb, U_min, U_max)  # 防止超出物理范围
    U_mat_fb[t] = U_fb

    # 3) 计算当前时刻的 Ω 和仿真出口浓度
    Omega_t = physical_omega(
        U_fb.reshape(1, -1), S_fb[t].reshape(1, -1), np.array([T_actual[t]]), opt_params
    )
    C_sim_fb[t] = C_in_mg[t] * np.exp(-Omega_t[0] / Q_actual[t])

    # 4) 更新误差与积分（用作下一时刻反馈）
    e_cur = C_sim_fb[t] - 50.0
    I_err += e_cur
    # 对积分项做限幅防止 windup
    I_err = np.clip(I_err, -50 / Ki if Ki > 0 else -100, 50 / Ki if Ki > 0 else 100)
    e_prev = e_cur

print("【有反馈】前馈+PI仿真结果")
print(f"  仿真出口浓度均值: {np.mean(C_sim_fb):.2f} mg/Nm³")
print(f"  仿真出口浓度标准差: {np.std(C_sim_fb):.4f}")
print(f"  中位数: {np.median(C_sim_fb):.2f}")
print(
    f"  5%分位: {np.percentile(C_sim_fb, 5):.2f}, 95%分位: {np.percentile(C_sim_fb, 95):.2f}"
)
print(f"\n实际出口浓度均值: {df['C_out_mgNm3'].mean():.2f}")
print(f"实际出口浓度标准差: {df['C_out_mgNm3'].std():.4f}")

# ===========================================================================
# 8. 可视化对比
# ===========================================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 无反馈浓度分布
axes[0, 0].hist(C_sim_ff, bins=60, edgecolor="k", alpha=0.7, color="steelblue")
axes[0, 0].axvline(50, color="r", linestyle="--", linewidth=2, label="目标 50")
axes[0, 0].axvline(
    np.mean(C_sim_ff), color="orange", label=f"均值={np.mean(C_sim_ff):.1f}"
)
axes[0, 0].set_xlabel("仿真出口浓度 (mg/Nm³)")
axes[0, 0].set_title("无反馈前馈仿真浓度分布")
axes[0, 0].legend()

# 有反馈浓度分布
axes[0, 1].hist(C_sim_fb, bins=60, edgecolor="k", alpha=0.7, color="seagreen")
axes[0, 1].axvline(50, color="r", linestyle="--", linewidth=2, label="目标 50")
axes[0, 1].axvline(
    np.mean(C_sim_fb), color="orange", label=f"均值={np.mean(C_sim_fb):.1f}"
)
axes[0, 1].set_xlabel("仿真出口浓度 (mg/Nm³)")
axes[0, 1].set_title("有反馈前馈+PI仿真浓度分布")
axes[0, 1].legend()

# 时间序列对比（前500点）
n_plt = min(500, len(df))
axes[0, 2].plot(
    df["C_out_mgNm3"].values[:n_plt], "b.", markersize=2, alpha=0.5, label="实际"
)
axes[0, 2].plot(C_sim_ff[:n_plt], "gray", alpha=0.6, linewidth=1, label="无反馈仿真")
axes[0, 2].plot(C_sim_fb[:n_plt], "r-", linewidth=1, label="有反馈仿真")
axes[0, 2].axhline(50, color="k", linestyle="--", alpha=0.5)
axes[0, 2].set_xlabel("样本序号")
axes[0, 2].set_ylabel("出口浓度 (mg/Nm³)")
axes[0, 2].set_title(f"前{n_plt}点仿真浓度对比")
axes[0, 2].legend()
axes[0, 2].grid(alpha=0.3)

# 电压对比（U1）
axes[1, 0].plot(U_mat_ff[:, 0], label="前馈电压 U1", alpha=0.7)
axes[1, 0].plot(U_mat_fb[:, 0], label="前馈+反馈 U1", alpha=0.7)
axes[1, 0].set_xlabel("样本序号")
axes[1, 0].set_ylabel("电压 (kV)")
axes[1, 0].set_title("电场1电压对比")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 反馈修正量 delta_U
delta_U_seq = U_mat_fb[:, 0] - U_mat_ff[:, 0]
axes[1, 1].plot(delta_U_seq, color="purple")
axes[1, 1].axhline(0, color="k", linestyle="--")
axes[1, 1].set_xlabel("样本序号")
axes[1, 1].set_ylabel("电压修正量 (kV)")
axes[1, 1].set_title("PI反馈电压修正量")
axes[1, 1].grid(alpha=0.3)

# 参数与统计信息
axes[1, 2].axis("off")
textstr = (
    f"辨识物理参数 (MSE损失):\n"
    f"K = {opt_params[0]:.1f}\nα = {opt_params[1]:.3f}\nβ = {opt_params[2]:.3f}\n"
)
for i in range(4):
    textstr += f"k{i+1} = {opt_params[3+i]:.5f}\n"
textstr += (
    f"\n无反馈仿真均值: {np.mean(C_sim_ff):.2f} mg/Nm³\n"
    f"标准差: {np.std(C_sim_ff):.4f}\n"
    f"有反馈仿真均值: {np.mean(C_sim_fb):.2f} mg/Nm³\n"
    f"标准差: {np.std(C_sim_fb):.4f}\n"
    f"PI参数: Kp={Kp}, Ki={Ki}"
)
axes[1, 2].text(
    0.05,
    0.5,
    textstr,
    fontsize=11,
    verticalalignment="center",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
)

plt.tight_layout()
plt.show()

print(
    "\n仿真完成。加入简单 PI 反馈后，出口浓度均值更接近 50 mg/Nm³，标准差进一步降低。"
)